# 01 · Fine-tune PhoWhisper-small cho môi trường ồn trong xe + đổi sang CTranslate2 int8

Cần **GPU T4**, bật Internet. Kaggle: Settings → Accelerator → GPU T4 x2. Colab: Runtime → Change runtime type → T4 GPU.
Secrets: `HF_TOKEN` (Kaggle: Add-ons → Secrets; Colab: biểu tượng chìa khoá bên trái, bật Notebook access).
Input tuỳ chọn: dataset riêng `xe-noise` gồm các file .wav tiếng ồn tự thu (gió + máy xe máy, điều hoà ô tô).
Trên Colab thì copy thư mục đó lên Drive: `MyDrive/ai-portfolio/viet-copilot/xe-noise/`.
Không có thì dùng tiếng ồn tổng hợp (brown/pink noise + tiếng ù tần số thấp giống động cơ).
Trên Colab checkpoint fine-tune (`ft/`) lưu trên Drive, bị ngắt thì chạy lại notebook là train tiếp.

Dữ liệu giọng: VIVOS (~15h, **CC BY-NC-SA 4.0**: chỉ dùng phi thương mại, model fine-tune cũng chịu điều kiện này).

Câu hỏi: PhoWhisper-small (WER VIVOS 6.33 theo paper) tụt bao nhiêu khi có tiếng ồn xe, và fine-tune với
tiếng ồn trộn ngẫu nhiên kéo lại được bao nhiêu mà không làm hỏng giọng sạch?

In [ ]:
import glob, os, sys
import numpy as np

IN_KAGGLE = os.path.exists("/kaggle/input")
IN_COLAB = "google.colab" in sys.modules
if IN_KAGGLE:
    CODE = glob.glob("/kaggle/input/**/copilot/__init__.py", recursive=True)[0].rsplit("/copilot", 1)[0]
    REPO = CODE.rsplit("/src", 1)[0]
    WORK = "/kaggle/working"
else:
    # Colab / máy local: lấy code từ GitHub, kết quả để trên Drive cho khỏi mất khi runtime bị ngắt
    REPO = "/content/viet-copilot" if IN_COLAB else os.path.abspath("viet-copilot")
    if not os.path.exists(REPO):
        !git clone --depth 1 https://github.com/DuongCodeAI/viet-copilot {REPO}
    CODE = f"{REPO}/src"
    if IN_COLAB:
        from google.colab import drive
        drive.mount("/content/drive")
    WORK = "/content/drive/MyDrive/ai-portfolio/viet-copilot" if IN_COLAB else os.getcwd()
os.makedirs(WORK, exist_ok=True)
os.chdir(WORK)
sys.path.insert(0, CODE)


def secret(name):
    # Kaggle Secrets / Colab Secrets / biến môi trường, thiếu thì trả None
    try:
        if IN_KAGGLE:
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret(name)
        if IN_COLAB:
            from google.colab import userdata
            return userdata.get(name)
    except Exception:
        pass
    return os.environ.get(name)

!pip install -q "transformers>=4.44" datasets jiwer soundfile librosa ctranslate2 accelerate evaluate
from copilot.speech import mix_noise

In [ ]:
from datasets import Audio, load_dataset

vivos = load_dataset("ademax/vivos-vie-speech2text")
print(vivos)
col_audio = [c for c in vivos["train"].column_names if "audio" in c][0]
col_text = [c for c in vivos["train"].column_names if c in ("sentence", "text", "transcription")][0]
vivos = vivos.cast_column(col_audio, Audio(sampling_rate=16000))

## Tiếng ồn

In [ ]:
import soundfile as sf
import librosa

noise_files = glob.glob("/kaggle/input/**/*.wav", recursive=True) + glob.glob(f"{WORK}/xe-noise/**/*.wav", recursive=True)
noises = [librosa.load(f, sr=16000)[0] for f in noise_files]
rng = np.random.default_rng(0)
if not noises:
    print("không có ồn thật -> tổng hợp")
    t = np.arange(16000 * 30) / 16000
    brown = np.cumsum(rng.normal(0, 1, len(t))); brown = brown - np.convolve(brown, np.ones(400) / 400, "same")
    hum = 0.3 * np.sin(2 * np.pi * 45 * t) + 0.15 * np.sin(2 * np.pi * 90 * t)  # máy nổ ~45Hz + hài
    noises = [(brown / np.abs(brown).max() + hum).astype(np.float32)]
print(len(noises), "đoạn ồn, tổng", round(sum(len(n) for n in noises) / 16000), "giây")
# giữ 20% ồn cho test, không dùng lúc train (tránh model "thuộc" đúng đoạn ồn đó)
noise_train = noises[: max(1, int(len(noises) * 0.8))]
noise_test = noises[int(len(noises) * 0.8):] or noises

In [ ]:
from transformers import WhisperForConditionalGeneration, WhisperProcessor

MID = "vinai/PhoWhisper-small"
proc = WhisperProcessor.from_pretrained(MID, language="vi", task="transcribe")
model = WhisperForConditionalGeneration.from_pretrained(MID)
model.generation_config.language = "vi"
model.generation_config.task = "transcribe"
model.generation_config.forced_decoder_ids = None

## Đánh giá trước fine-tune: sạch và có ồn (SNR 10 / 5 / 0 dB)

In [ ]:
import jiwer, torch, re

def norm(s):
    return re.sub(r"[^\w\s]", "", s.lower()).strip()

test = vivos["test"].select(range(400))

@torch.no_grad()
def wer_on(m, snr=None, bs=16):
    m.eval().cuda()
    refs, hyps = [], []
    r = np.random.default_rng(123)
    for i in range(0, len(test), bs):
        batch = test[i:i + bs]
        audios = [a["array"].astype(np.float32) for a in batch[col_audio]]
        if snr is not None:
            audios = [mix_noise(a, noise_test[r.integers(len(noise_test))], snr, r) for a in audios]
        feats = proc.feature_extractor(audios, sampling_rate=16000, return_tensors="pt").input_features.cuda().half()
        with torch.autocast("cuda"):
            ids = m.generate(feats, max_new_tokens=128)
        hyps += [norm(t) for t in proc.batch_decode(ids, skip_special_tokens=True)]
        refs += [norm(t) for t in batch[col_text]]
    return round(100 * jiwer.wer(refs, hyps), 2)

before = {f"snr{s}" if s is not None else "sạch": wer_on(model, s) for s in (None, 10, 5, 0)}
before

## Fine-tune: mỗi mẫu train 70% bị trộn ồn với SNR ngẫu nhiên 0-15 dB, 30% giữ sạch
Giữ một phần sạch để không quên giọng sạch. Train ít epoch, lr nhỏ: PhoWhisper đã tốt sẵn.

In [ ]:
from dataclasses import dataclass
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments

train = vivos["train"]

def prep(batch, rng=np.random.default_rng(1)):
    a = batch[col_audio]["array"].astype(np.float32)
    if rng.random() < 0.7:
        a = mix_noise(a, noise_train[rng.integers(len(noise_train))], rng.uniform(0, 15), rng)
    batch["input_features"] = proc.feature_extractor(a, sampling_rate=16000).input_features[0]
    batch["labels"] = proc.tokenizer(batch[col_text]).input_ids
    return batch

train = train.map(prep, remove_columns=train.column_names, num_proc=2)

@dataclass
class Collator:
    def __call__(self, feats):
        inp = proc.feature_extractor.pad([{"input_features": f["input_features"]} for f in feats], return_tensors="pt")
        lab = proc.tokenizer.pad([{"input_ids": f["labels"]} for f in feats], return_tensors="pt")
        labels = lab["input_ids"].masked_fill(lab.attention_mask.ne(1), -100)
        if (labels[:, 0] == model.generation_config.decoder_start_token_id).all():
            labels = labels[:, 1:]
        inp["labels"] = labels
        return inp

args = Seq2SeqTrainingArguments(output_dir="ft", per_device_train_batch_size=16, gradient_accumulation_steps=2,
                                learning_rate=1e-5, warmup_steps=200, num_train_epochs=2, fp16=True,
                                gradient_checkpointing=True, logging_steps=50, report_to=[],
                                # Colab có thể ngắt giữa chừng -> checkpoint lên Drive (~3GB/lần, giữ 1 bản)
                                save_strategy="steps" if IN_COLAB else "no", save_steps=200, save_total_limit=1)
trainer = Seq2SeqTrainer(model=model, args=args, train_dataset=train, data_collator=Collator())
trainer.train(resume_from_checkpoint=bool(glob.glob("ft/checkpoint-*")))

In [ ]:
after = {f"snr{s}" if s is not None else "sạch": wer_on(model, s) for s in (None, 10, 5, 0)}
import pandas as pd
pd.DataFrame({"PhoWhisper-small gốc": before, "fine-tune có ồn": after})

## Đổi sang CTranslate2 int8 (chạy bằng faster-whisper trên laptop) + đẩy lên HF Hub

In [ ]:
model.save_pretrained("phowhisper-noisy"); proc.save_pretrained("phowhisper-noisy")
!ct2-transformers-converter --model phowhisper-noisy --output_dir phowhisper-noisy-ct2-int8 --quantization int8 \
    --copy_files tokenizer.json preprocessor_config.json
!du -sh phowhisper-noisy-ct2-int8

from huggingface_hub import HfApi
api = HfApi(token=secret("HF_TOKEN"))
REPO = "DuongCodeAI/phowhisper-small-noisy-ct2-int8"
api.create_repo(REPO, exist_ok=True)
api.upload_folder(folder_path="phowhisper-noisy-ct2-int8", repo_id=REPO)
import json; json.dump({"before": before, "after": after}, open("wer.json", "w"), ensure_ascii=False)
api.upload_file(path_or_fileobj="wer.json", path_in_repo="wer.json", repo_id=REPO)